**Import bibliotek i utworzenie SparkSession**

Utworzono lokalną sesję Spark działającą w trybie local[*]. Wszystkie dostępne rdzenie procesora są wykorzystywane jako lokalny odpowiednik klastra Databricks.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("FASTQ_Analysis")
    .master("local[*]")
    .getOrCreate()
)

spark

**7.1 Wczytanie i parsowanie pliku FASTQ**

Plik FASTQ został wczytany jako DataFrame tekstowy. Każda linia pliku stanowi osobny rekord. Ponieważ pojedynczy odczyt FASTQ składa się z 4 linii, liczba odczytów została wyznaczona jako liczba wszystkich linii podzielona przez 4. Plik SRR16356247_1_1.fastq powstał poprzez wyekstrahowanie pierwszych 100 odczytów z oryginalnego pliku: 
`head -n 400 SRR16356247_1.fastq > SRR16356247_1_1.fastq`

In [2]:
file_path = "/home/clusters/work/SRR16356247_1_1.fastq"

lines_df = spark.read.text(file_path)

lines_df.show(8, truncate=False)

print(f"Liczba linii: {lines_df.count()}")
print(f"Liczba odczytów: {lines_df.count() // 4}")

+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                  |
+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|@SRR16356247.1 1 length=151                                                                                                                            |
|GCTCCCAACCAAGCTCTNTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGNGCTGGGCTCCNGTGCGTTCGGCACGGTGTATAAGGTAAGGTCCCTGGCACAGGCCTCTGGGCTGGGCCGCAGGGCCTCTCATGGTCTGGTGG|
|+SRR16356247.1 1 length=151                                                                                                                            |
|A=AFFFFFFFFFFFFFF#F/FAFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFF#FFFFFFFFFF#FFFFF

**7.2 Dodanie numerów linii**

Za pomocą funkcji `monotonically_increasing_id()` nadano każdemu wierszowi w DataFrame unikalny, rosnący numer. 

In [3]:
lines_with_id = lines_df.withColumn(
    "line_number",
    monotonically_increasing_id()
)

lines_with_id.show(8, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|value                                                                                                                                                  |line_number|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|@SRR16356247.1 1 length=151                                                                                                                            |0          |
|GCTCCCAACCAAGCTCTNTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGNGCTGGGCTCCNGTGCGTTCGGCACGGTGTATAAGGTAAGGTCCCTGGCACAGGCCTCTGGGCTGGGCCGCAGGGCCTCTCATGGTCTGGTGG|1          |
|+SRR16356247.1 1 length=151                                                                                                                            |2          |
|A=A

In [4]:
window = Window.orderBy(monotonically_increasing_id())

lines_with_id = lines_df.withColumn(
    "line_number",
    row_number().over(window) - 1
)

lines_with_id.show(8, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|value                                                                                                                                                  |line_number|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|@SRR16356247.1 1 length=151                                                                                                                            |0          |
|GCTCCCAACCAAGCTCTNTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGNGCTGGGCTCCNGTGCGTTCGGCACGGTGTATAAGGTAAGGTCCCTGGCACAGGCCTCTGGGCTGGGCCGCAGGGCCTCTCATGGTCTGGTGG|1          |
|+SRR16356247.1 1 length=151                                                                                                                            |2          |
|A=A

**7.3 Określenie typu linii w FASTQ**

Przypisano etykiety kolejnym wierszom na podstawie ich funkcji.

In [5]:
lines_typed = lines_with_id.withColumn(
    "line_type",
    when(col("line_number") % 4 == 0, "header")
    .when(col("line_number") % 4 == 1, "sequence")
    .when(col("line_number") % 4 == 2, "separator")
    .when(col("line_number") % 4 == 3, "quality")
)

lines_typed.show(12, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+
|value                                                                                                                                                  |line_number|line_type|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+
|@SRR16356247.1 1 length=151                                                                                                                            |0          |header   |
|GCTCCCAACCAAGCTCTNTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGNGCTGGGCTCCNGTGCGTTCGGCACGGTGTATAAGGTAAGGTCCCTGGCACAGGCCTCTGGGCTGGGCCGCAGGGCCTCTCATGGTCTGGTGG|1          |sequence |
|+SRR16356247.1 1 length=151                                                                                            

**7.4 Utworzenie identyfikatora odczytu**

Nadanie wszysktim wierszom należacym do jednego rekordu tego samego `record_id`.

In [6]:
lines_with_record = lines_typed.withColumn(
    "record_id",
    (col("line_number") / 4).cast("integer")
)

lines_with_record.show(16, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+---------+
|value                                                                                                                                                  |line_number|line_type|record_id|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+---------+
|@SRR16356247.1 1 length=151                                                                                                                            |0          |header   |0        |
|GCTCCCAACCAAGCTCTNTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGNGCTGGGCTCCNGTGCGTTCGGCACGGTGTATAAGGTAAGGTCCCTGGCACAGGCCTCTGGGCTGGGCCGCAGGGCCTCTCATGGTCTGGTGG|1          |sequence |0        |
|+SRR16356247.1 1 length=151                                          

**7.5 Przekształcenie do formatu szerokiego (Pivot)**

In [7]:
fastq_wide = lines_with_record.groupBy("record_id").pivot("line_type").agg(
    first("value")
)

fastq_wide.show(5, truncate=False)
fastq_wide.printSchema()

+---------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|record_id|header                     |quality                                                                                                                                                |separator                  |sequence                                                                                                                                               |
+---------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------+-------------------------------------

**7.6 Czyszczenie danych nagłówka**

In [8]:
fastq_clean = fastq_wide.withColumn(
    "read_id",
    split(
        regexp_replace(col("header"), "^@", ""),
        " "
    )[0]
)

fastq_clean.select(
    "record_id",
    "read_id",
    "sequence",
    "quality"
).show(5, truncate=False)

+---------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|record_id|read_id      |sequence                                                                                                                                               |quality                                                                                                                                                |
+---------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|0        

In [9]:
fastq_final = fastq_clean.select(
    "record_id",
    "read_id",
    "sequence",
    "quality"
)

fastq_final.cache()

fastq_final.count()

100

**Zadanie 3. Kompleksowa analiza i optymalizacja**

In [10]:
quality_analysis = fastq_final.withColumn(
    "has_low_quality",
    instr(col("quality"), "!") > 0
)

In [11]:
quality_analysis = quality_analysis.withColumn(
    "seq_length",
    length(col("sequence"))
)

In [12]:
quality_summary = quality_analysis.groupBy(
    "has_low_quality",
    "seq_length"
).count()

In [13]:
quality_summary = quality_summary.orderBy(
    "seq_length"
)

In [14]:
quality_summary.cache()
quality_summary.count()

1

In [15]:
quality_summary.count()

1

In [16]:
quality_summary.select("seq_length").distinct().count()

1

In [17]:
quality_summary.show()

+---------------+----------+-----+
|has_low_quality|seq_length|count|
+---------------+----------+-----+
|          false|       151|  100|
+---------------+----------+-----+



**Ile Jobs zostało utworzonych? Wypiszmy każdą akcję z kodu i przypisz jej numer Joba z
UI.**

| Akcja w kodzie                                                 | Job w Spark UI | Co zrobił Spark                                                                                                          |
| -------------------------------------------------------------- | -------------- | ------------------------------------------------------------------------------------------------------------------------ |
| `quality_summary.cache()` + pierwsze `quality_summary.count()` | **Job 20**     | Pierwsze wykonanie zapytania. Spark policzył agregację `groupBy().count()`, wykonał sortowanie i zapisał wynik do cache. |
| drugie `quality_summary.count()`                               | **Job 21**     | Spark użył już danych z cache (`InMemoryTableScan`). Nie musiał ponownie wykonywać wcześniejszych transformacji.         |
| trzecie `quality_summary.count()`                              | **Job 22**     | Ponownie odczytał wynik z cache. Widać `InMemoryTableScan`.                              |
| `quality_summary.select("seq_length").distinct().count()`      | **Job 23**     | Odczytał cache i wykonał dodatkową operację `distinct()` na kolumnie `seq_length`, a następnie policzył wynik.           |
| `quality_summary.show()`                                       | **Job 24**     | Odczytał cache i przygotował wynik do wyświetlenia.                                                                      |


**Narysujmy schemat Stage'ów dla pierwszego Joba (tego z groupBy):**
* **Stage 1: Jakie operacje się tu odbywały? Czy widzimy w metrykach Shuffle
Write? (to etap, na którym taski przygotowują dane do agregacji).**
* **Stage 2: Jakie operacje? Czy widzimy Shuffle Read? (to etap agregacji, na
którym taski czytają dane przygotowane przez Stage 1).**

W przypadku Job 20 Spark nie utworzył dwóch osobnych Stage'ów widocznych w UI. Operacje odpowiadające przygotowaniu danych do agregacji oraz końcowej agregacji zostały połączone w jeden Stage (Stage 30). W DAG widoczne są operacje HashAggregate, Exchange oraz ShuffledRowRDD, które wskazują na wykonanie shuffle. Stage 29 został pominięty, ponieważ dane były pobierane z cache (InMemoryTableScan), a nie ponownie odczytywane ze źródła.

**Które Jobs wykorzystały cache? Zidentyfikujmy Joby, których etapy zostały pominięte
("skipped"). Dlaczego tak się stało?**

| Job | Akcja w kodzie                                               |
| --- | ------------------------------------------------------------ |
| 20  | `quality_summary.count()` (pierwsze wykonanie po `.cache()`) |
| 21  | `quality_summary.count()` (drugie wywołanie)                 |
| 22  | `quality_summary.count()` (kolejne wywołanie)                |
| 23  | `quality_summary.select("seq_length").distinct().count()`    |
| 24  | `quality_summary.show()`                                     |

Spark pominął Stage oznaczone jako skipped, ponieważ dane były dostępne w cache (InMemoryTableScan).

**Porównajmy Input Size między różnymi Jobami. Dlaczego Jobs korzystające z cache
mają bardzo mały lub zerowy rozmiar wejściowy?**

Joby korzystające z cache mają bardzo mały lub zerowy Input Size, ponieważ nie odczytują ponownie danych ze źródła (np. pliku), tylko przetwarzają już przygotowany wynik.

**Efektywność cache: porównajmy czas wykonania pierwszego Joba (który musiał
obliczyć wszystko od zera) z czasami kolejnych Jobów (które czytały z cache). Jaki jest
zysk?**


Pierwszy Job, który musiał wykonać pełny pipeline transformacji i zapisać wynik do cache, trwał dłużej (około 0,2 s, Job 20). Kolejne Joby korzystające z cache wykonywały się szybciej: Job 21 - 29 ms, Job 22 - 38 ms, Job 23 - 79 ms, Job 24 - 91 ms. Zysk wynika z tego, że Spark nie musiał ponownie wykonywać wcześniejszych transformacji ani odczytywać danych źródłowych, tylko korzystał z już przygotowanego wyniku zapisanego w pamięci.

**Locality Level: Czy wszystkie taski w Jobach korzystających z cache miały
PROCESS_LOCAL? Jeśli nie, spróbujmy określić, dlaczego niektóre taski musiały
czytać dane z pamięci na innym węźle (NODE_LOCAL)**

Wszystkie taski w Jobach od Job 20 do Job 24 miały poziom lokalności PROCESS_LOCAL. Spark odczytywał dane z cache znajdującego się lokalnie na tym samym executorze, bez konieczności przesyłania danych między węzłami (NODE_LOCAL).